# Lab 1: Building RelWeights

This notebook turns the six-step RelWeights construction into executable matrix checks. It follows Sections 1 through 6 of the local reference note and uses the worked example from the [NYU proposal](../../assets/papers/nyu_proposal_final.pdf). The formal reference is the [RelWeights Laplacian note](../../assets/papers/laplacians_spatial_operators_relweights.pdf).

## Step 1: Incidence matrices

The construction starts with an incidence matrix $B$ that records how analysis units $\alpha_i$ intersect inherited supports $\beta_k$.

- $B_{\mathrm{bin}}$ records whether an overlap exists
- $B_{\mathrm{area}}$ records the share of each $\alpha_i$ carried by each support $\beta_k$

For area-overlap incidence, each row sums to one. The binary matrix is just the support pattern of the area-overlap matrix.

In [1]:
import numpy as np

B_area = np.array(
    [
        [0.60, 0.40, 0.00, 0.00],
        [0.30, 0.40, 0.30, 0.00],
        [0.00, 0.20, 0.50, 0.30],
        [0.00, 0.00, 0.55, 0.45],
    ]
)

B_bin = (B_area > 0).astype(int)

print("B_area =")
print(B_area)
print("\nrow sums =", B_area.sum(axis=1))
print("\nB_bin =")
print(B_bin)

B_area =
[[0.6  0.4  0.   0.  ]
 [0.3  0.4  0.3  0.  ]
 [0.   0.2  0.5  0.3 ]
 [0.   0.   0.55 0.45]]

row sums = [1. 1. 1. 1.]

B_bin =
[[1 1 0 0]
 [1 1 1 0]
 [0 1 1 1]
 [0 0 1 1]]


## Step 2: Proposal example for $B$, $R$, and $L_R$

The proposal uses four districts $A, B, C, D$ and four contextual regimes $X, Y, Z, W$.

- $A$ intersects $X, Y$
- $B$ intersects $X, Y, Z$
- $C$ intersects $W, Y, Z$
- $D$ intersects $W, Z$

This yields a binary incidence matrix $B$. The Gram matrix $B B^\top$ counts shared supports. Removing the diagonal gives the RelWeights matrix $R$.

In [2]:
labels = ["A", "B", "C", "D"]
regimes = ["X", "Y", "Z", "W"]

B = np.array(
    [
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [0, 1, 1, 1],
        [0, 0, 1, 1],
    ],
    dtype=float,
)

gram = B @ B.T
R = gram - np.diag(np.diag(gram))
D_R = np.diag(R.sum(axis=1))
L_R = D_R - R

print("B =")
print(B.astype(int))
print("\nB B^T =")
print(gram.astype(int))
print("\nR =")
print(R.astype(int))
print("\nD_R =")
print(D_R.astype(int))
print("\nL_R =")
print(L_R.astype(int))

B =
[[1 1 0 0]
 [1 1 1 0]
 [0 1 1 1]
 [0 0 1 1]]

B B^T =
[[2 2 1 0]
 [2 3 2 1]
 [1 2 3 2]
 [0 1 2 2]]

R =
[[0 2 1 0]
 [2 0 2 1]
 [1 2 0 2]
 [0 1 2 0]]

D_R =
[[3 0 0 0]
 [0 5 0 0]
 [0 0 5 0]
 [0 0 0 3]]

L_R =
[[ 3 -2 -1  0]
 [-2  5 -2 -1]
 [-1 -2  5 -2]
 [ 0 -1 -2  3]]


## Step 3: The Laplacian as relational roughness

The core identity from the reference note is

$$
x^\top L_R x = \frac{1}{2} \sum_{i,j} R_{ij}(x_i - x_j)^2
$$

So the quadratic form is an energy functional. It becomes large when strongly related units carry very different values.

In [3]:
x = np.array([1.0, 2.5, -0.5, 0.0])

lhs = float(x @ L_R @ x)
rhs = 0.5 * sum(
    R[i, j] * (x[i] - x[j]) ** 2
    for i in range(len(x))
    for j in range(len(x))
)

print("x =", x)
print("x^T L_R x =", lhs)
print("roughness sum =", rhs)
print("identity holds:", np.isclose(lhs, rhs))

x = [ 1.   2.5 -0.5  0. ]
x^T L_R x = 31.5
roughness sum = 31.5
identity holds: True


## Step 4: Positive semidefiniteness and nullspace

Because the quadratic form is a weighted sum of squared pairwise differences, $L_R$ is positive semidefinite. The constant vector also lies in the nullspace, so the zero mode represents no relational curvature.

In [4]:
eigvals = np.linalg.eigvalsh(L_R)
null_test = L_R @ np.ones(L_R.shape[0])

print("eigenvalues =", np.round(eigvals, 6))
print("\nL_R @ 1 =", null_test)
print("\nall eigenvalues nonnegative:", np.all(eigvals >= -1e-10))

eigenvalues = [0.       2.763932 6.       7.236068]

L_R @ 1 = [0. 0. 0. 0.]

all eigenvalues nonnegative: True


## Step 5: Binary versus area-overlap support

The same pipeline works with $B_{\mathrm{area}}$. Binary support counts how many contextual regimes are shared. Area-overlap support weights those similarities by how much of each analysis unit is carried by each regime.

In [5]:
def build_relweights(B_matrix):
    gram_matrix = B_matrix @ B_matrix.T
    R_matrix = gram_matrix - np.diag(np.diag(gram_matrix))
    D_matrix = np.diag(R_matrix.sum(axis=1))
    L_matrix = D_matrix - R_matrix
    return gram_matrix, R_matrix, D_matrix, L_matrix

_, R_binary, _, _ = build_relweights(B_bin.astype(float))
_, R_area, _, _ = build_relweights(B_area)

print("R from binary support =")
print(np.round(R_binary, 3))
print("\nR from area-overlap support =")
print(np.round(R_area, 3))

R from binary support =
[[0. 2. 1. 0.]
 [2. 0. 2. 1.]
 [1. 2. 0. 2.]
 [0. 1. 2. 0.]]

R from area-overlap support =
[[0.    0.34  0.08  0.   ]
 [0.34  0.    0.23  0.165]
 [0.08  0.23  0.    0.41 ]
 [0.    0.165 0.41  0.   ]]


## Step 6: Module order

This foundations notebook feeds the rest of the lab. The same operator state will be reused across build, export, spectral, diagnostics, externalities, interpolation, and simulation.

In [6]:
module_sequence = [
    "build",
    "export",
    "spectral",
    "diagnostics",
    "externalities",
    "interpolation",
    "simulate",
]

module_sequence

['build',
 'export',
 'spectral',
 'diagnostics',
 'externalities',
 'interpolation',
 'simulate']

## Appendix: geometry sandbox

The cells below are preserved from the earlier notebook stub so the lattice plotting work remains available while the theory notebook grows.

In [7]:
class PolygonShape:
    def __init__(self, shape_id, vectors, color='blue', value=None):
        self.id = shape_id
        # Vectors should be a list of (x, y) tuples or lists defining the polygon vertices
        self.vectors = vectors
        self.color = color
        self.value = value # New attribute for numerical data
        self.centroid = self._calculate_centroid()

    def _calculate_centroid(self):
        if not self.vectors:
            return (0, 0) # Default for an empty polygon

        x_coords = [p[0] for p in self.vectors]
        y_coords = [p[1] for p in self.vectors]

        # Simple centroid calculation for a polygon (average of vertices)
        # For more complex polygons, a more robust method might be needed,
        # but for simple shapes like rectangles, this is usually sufficient.
        centroid_x = sum(x_coords) / len(self.vectors)
        centroid_y = sum(y_coords) / len(self.vectors)
        return (centroid_x, centroid_y)

    def __repr__(self):
        return f"PolygonShape(id='{self.id}', vectors={self.vectors}, color='{self.color}', value={self.value}, centroid={self.centroid})"

class LatticeGrid:
    def __init__(self, width, height, name=None):
        self.width = width
        self.height = height
        self.name = name if name is not None else f"LatticeGrid_{width}x{height}"
        # Initialize the grid with empty lists, where each list represents a 'stack' or 'layer' of shapes
        self.grid = [[[] for _ in range(width)] for _ in range(height)]

    def assign_shape(self, x, y, shape_object, layer=None):
        # x, y are grid coordinates (0-indexed)
        # shape_object should be an instance of PolygonShape

        if not (0 <= x < self.width and 0 <= y < self.height):
            raise ValueError("Coordinates out of grid bounds.")

        if not isinstance(shape_object, PolygonShape):
            raise TypeError("shape_object must be an instance of PolygonShape.")

        if layer is None:
            # Add to the top layer by default
            self.grid[y][x].append(shape_object)
        else:
            # Insert at a specific layer index, or append if layer is out of bounds
            if layer < len(self.grid[y][x]):
                self.grid[y][x].insert(layer, shape_object)
            else:
                # If layer is beyond current layers, append
                self.grid[y][x].append(shape_object)

    def get_shapes_at(self, x, y):
        if not (0 <= x < self.width and 0 <= y < self.height):
            raise ValueError("Coordinates out of grid bounds.")
        return self.grid[y][x]

    def display_grid_summary(self):
        print("Lattice Grid Summary:")
        print(f"Width: {self.width}, Height: {self.height}")
        for y in range(self.height):
            row_summary = []
            for x in range(self.width):
                num_shapes = len(self.grid[y][x])
                row_summary.append(f"({x},{y}):{num_shapes}")
            print(" | ".join(row_summary))

In [8]:
import plotly.graph_objects as go
import plotly.express as px # For continuous color scales

def plot_lattice_grid(*grid_instances):
    if not grid_instances:
        print("No grid instances provided to plot.")
        return

    fig = go.Figure()

    # Calculate overall maximum width and height across all grids
    max_width = 0
    max_height = 0
    for grid_instance in grid_instances:
        max_width = max(max_width, grid_instance.width)
        max_height = max(max_height, grid_instance.height)

    # Collect all shape values from all grids to determine the global color scale range
    all_shape_values = []
    for grid_instance in grid_instances:
        for y in range(grid_instance.height):
            for x in range(grid_instance.width):
                shapes_in_cell = grid_instance.get_shapes_at(x, y)
                for shape in shapes_in_cell:
                    if shape.value is not None:
                        all_shape_values.append(shape.value)

    min_val = min(all_shape_values) if all_shape_values else 0
    max_val = max(all_shape_values) if all_shape_values else 1

    color_scale = px.colors.sequential.Viridis

    legends_shown_for_grid = set() # Keep track of which grid legends have been shown

    for grid_idx, grid_instance in enumerate(grid_instances):
        # Use grid_instance.name for legend if available, otherwise fall back to default
        grid_legend_name = grid_instance.name if hasattr(grid_instance, 'name') else f'Grid {grid_idx}'

        # Flag to control showing legend for this specific grid
        show_this_grid_legend = False
        if grid_legend_name not in legends_shown_for_grid:
            show_this_grid_legend = True
            legends_shown_for_grid.add(grid_legend_name) # Mark as shown

        for y in range(grid_instance.height):
            for x in range(grid_instance.width):
                shapes_in_cell = grid_instance.get_shapes_at(x, y)

                for shape_idx, shape in enumerate(shapes_in_cell):
                    # Determine color based on shape.value and the color scale
                    if shape.value is not None:
                        normalized_value = (shape.value - min_val) / (max_val - min_val) if (max_val - min_val) > 0 else 0.5
                        color_index = int(normalized_value * (len(color_scale) - 1))
                        shape_color = color_scale[color_index]
                    else:
                        shape_color = shape.color # Fallback to original color if no value

                    # Translate local shape vectors to grid coordinates,
                    # aligning the shape's centroid with the cell's center (x, y)
                    centroid_offset_x = x - shape.centroid[0]
                    centroid_offset_y = y - shape.centroid[1]

                    global_x_coords = [p[0] + centroid_offset_x for p in shape.vectors]
                    global_y_coords = [p[1] + centroid_offset_y for p in shape.vectors]

                    fig.add_trace(go.Scatter(
                        x=global_x_coords + [global_x_coords[0]], # Close the polygon
                        y=global_y_coords + [global_y_coords[0]], # Close the polygon
                        mode='lines',
                        fill='toself',
                        fillcolor=shape_color, # Use dynamically determined color
                        line=dict(color=shape_color, width=1), # Line color also matches fill
                        # Set name and legendgroup to allow toggling entire grid layers
                        name=grid_legend_name,
                        legendgroup=grid_legend_name,
                        showlegend=show_this_grid_legend, # Use the new flag
                        hoverinfo='text',
                        text=f'Grid: {grid_idx}<br>ID: {shape.id}<br>Value: {shape.value}<br>Cell: ({x},{y})'
                    ))
                    # After the first trace for a grid, ensure subsequent traces from the same grid don't show their own legend entry
                    show_this_grid_legend = False

    # Update layout to represent a grid with integer coordinates as midpoints
    fig.update_layout(
        title=f'Lattice Grid with Polygons (Max {max_width}x{max_height}) - Multiple Layers',
        xaxis=dict(
            range=[-0.5, max_width - 0.5],
            tickvals=list(range(max_width)),
            gridcolor='darkgray',
            zeroline=False,
            title=None, # Removed x-axis title
            minor=dict(
                tickmode='auto',
                dtick=1,
                gridcolor='lightgray',
                griddash='dot',
                showgrid=True
            )
        ),
        yaxis=dict(
            range=[max_height - 0.5, -0.5],
            tickvals=list(range(max_height)),
            gridcolor='darkgray',
            zeroline=False,
            scaleratio=1,
            title=None, # Removed y-axis title
            minor=dict(
                tickmode='auto',
                dtick=1,
                gridcolor='lightgray',
                griddash='dot',
                showgrid=True
            )
        ),
        height=50 + max_height * 100,
        width=50 + max_width * 100,
        showlegend=True
    )

    fig.show()